# Step 10c-3: Qwen2.5-7B-Instruct chấm 4 bộ mới (SciFact, NFCorpus, DBPedia-entity, FEVER)

LLM chấm thứ hai, mã nguồn mở, lặp lại đúng các chế độ gpt-4o-mini đã chấm ở Step 10c-2 trên tập test (không chấm train, giống Step 10-B1). Dùng nguyên `llm_score.py --backend vllm` của Step 10-B1

| Ô | Chế độ | Tập | Số prompt |
|---|---|---|---|
| A | `paper`, `code`, `paper_fixed`, `code_shuffle`, `dat` | test SciFact, NFCorpus, DBPedia-entity | 5 × 1.023 |
| B | `paper`, `dat` | FEVER test (6.666 query) | 2 × 6.666 |
| C | `code`, `paper_fixed`, `code_shuffle` (`paper` lấy từ cache ô B) | 1.500 query đầu của FEVER test | 3 × 1.500 |

- **Accelerator:** GPU T4 ×2. **Internet:** On. **Secret:** `HF_TOKEN` 
- **Input:** `qudar-beir-data-2`, `qudar-runs-2`, `qudar-patches`
- **Thời gian ước tính:** cài vLLM 5–8 phút; 3 lần nạp mô hình × 2–3 phút; khoảng 22.900 prompt ở 8–12 prompt/giây (như Step 10-B1) → 30–50 phút; đọc file FEVER 2 lần × 5–8 phút. **Tổng khoảng 1–1,5 giờ GPU.**
- **Cách chạy:** Save Version → **Save & Run All (Commit)**.

In [1]:
import glob, json, os
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

def first(pattern, what):
    hits = sorted(glob.glob(pattern, recursive=True))
    assert hits, f"không tìm thấy {what} ({pattern}): kiểm tra dataset đã gắn chưa"
    return hits[0]
DATA2 = os.path.dirname(os.path.dirname(first("/kaggle/input/**/qudar-beir-data-2/**/dbpedia-entity/corpus.jsonl", "qudar-beir-data-2")))
PATCH = os.path.dirname(first("/kaggle/input/**/llm_score.py", "qudar-patches"))
RUNS  = os.path.dirname(os.path.dirname(first("/kaggle/input/**/fever_test/retrieval_result.json", "qudar-runs-2")))
TEST  = ["scifact_test", "nfcorpus_test", "dbpedia-entity_test", "fever_test"]
missing = [n for n in TEST if not os.path.exists(f"{RUNS}/{n}/retrieval_result.json")]
assert not missing, f"qudar-runs-2 thiếu: {missing}"
assert "--backend" in open(f"{PATCH}/llm_score.py").read(), "qudar-patches chưa phải bản mới"
print("DATA2 =", DATA2, "\nPATCH =", PATCH, "\nRUNS  =", RUNS)
!pip install -q vllm pytrec-eval-terrier
!pip install -q --no-deps beir
!git clone -q https://github.com/kaist-dmlab/QuDAR.git /kaggle/working/QuDAR
# Không pip install QuDAR: chỉ cần thư mục src, tránh đụng phiên bản torch/transformers của vLLM
OUT = "/kaggle/working/llm_qwen"
L = (f"python {PATCH}/llm_score.py --repo /kaggle/working/QuDAR --runs {RUNS} --data-root {DATA2} "
     f"--out {OUT} --backend vllm --model Qwen/Qwen2.5-7B-Instruct")

DATA2 = /kaggle/input/datasets/karrychuong/qudar-beir-data-2/data 
PATCH = /kaggle/input/datasets/karrychuong/qudar-patches/qudar-patches 
RUNS  = /kaggle/input/datasets/karrychuong/qudar-runs-2/runs
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.65.0 which is incompatible.
litellm 1.109.1 requires huggingface-hub<2.0,>=0.34.0, but you have huggingface-hub 2.4.0 which is incompatible.
numba-cuda 0.28.2 requires cuda-core<1.0.0,>=0.5.1, but you have cuda-core 1.3.1 which is incompatible.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.4.0 which is incompatible.
cuml-cu13 26.6.0 requires numba<0.65.0,>=0.60.0, but you have numba 0.65.0 which is incompatible.
cudf-cu13 26.6.0 requires numba<0.65.0,>=0.60.0, but you have numba 0.65.0 which is incompatible.


In [2]:
# A. Ba bộ nhỏ: cả 5 chế độ trong 1 lần nạp mô hình
!{L} --mode paper code paper_fixed code_shuffle dat --names scifact_test nfcorpus_test dbpedia-entity_test

W1124 15:54:04.256000 90 torch/_opaque_base.py:6] torch._opaque_base is deprecated, use torch._custom_class_base instead
W1124 15:54:04.257000 90 torch/_library/opaque_object.py:288] register_opaque_type is deprecated, use register_custom_class instead
W1124 15:54:04.257000 90 torch/_library/opaque_object.py:211] typ='value' is deprecated, use typ='constant' instead
(MainProcess pid=90) INFO 11-24 15:54:30 [startup.py:96] non-default args: {'dtype': 'half', 'max_model_len': 6144, 'tensor_parallel_size': 2, 'gpu_memory_utilization': 0.9, 'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-7B-Instruct'}
config.json: 100%|█████████████████████████████| 663/663 [00:00<00:00, 2.09MB/s]
(MainProcess pid=90) INFO 11-24 15:54:47 [model.py:761] Resolved architecture: Qwen2ForCausalLM
(MainProcess pid=90) WARNING 11-24 15:54:47 [model.py:2560] Casting torch.bfloat16 to torch.float16.
(MainProcess pid=90) INFO 11-24 15:54:47 [model.py:2214] Using max model len 6144
(MainProcess pid=90) INFO 11-24 1

In [3]:
# B. FEVER test đủ 6.666 query: QuDAR-llm (prompt paper) và DAT cho bảng chính
!{L} --mode paper dat --names fever_test

W1124 16:41:40.183000 553 torch/_opaque_base.py:6] torch._opaque_base is deprecated, use torch._custom_class_base instead
W1124 16:41:40.184000 553 torch/_library/opaque_object.py:288] register_opaque_type is deprecated, use register_custom_class instead
W1124 16:41:40.184000 553 torch/_library/opaque_object.py:211] typ='value' is deprecated, use typ='constant' instead
(MainProcess pid=553) INFO 11-24 16:41:55 [startup.py:96] non-default args: {'dtype': 'half', 'max_model_len': 6144, 'tensor_parallel_size': 2, 'gpu_memory_utilization': 0.9, 'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-7B-Instruct'}
(MainProcess pid=553) INFO 11-24 16:41:55 [model.py:761] Resolved architecture: Qwen2ForCausalLM
(MainProcess pid=553) WARNING 11-24 16:41:55 [model.py:2560] Casting torch.bfloat16 to torch.float16.
(MainProcess pid=553) INFO 11-24 16:41:55 [model.py:2214] Using max model len 6144
(MainProcess pid=553) INFO 11-24 16:41:55 [scheduler.py:354] Chunked prefill is enabled with max_num_batche

In [4]:
# C. Ablation trên 1.500 query đầu của FEVER (chế độ paper lấy từ cache của ô B)
!{L} --mode paper code paper_fixed code_shuffle --names fever_test --max-queries 1500

W1124 17:54:40.854000 840 torch/_opaque_base.py:6] torch._opaque_base is deprecated, use torch._custom_class_base instead
W1124 17:54:40.855000 840 torch/_library/opaque_object.py:288] register_opaque_type is deprecated, use register_custom_class instead
W1124 17:54:40.856000 840 torch/_library/opaque_object.py:211] typ='value' is deprecated, use typ='constant' instead
(MainProcess pid=840) INFO 11-24 17:55:02 [startup.py:96] non-default args: {'dtype': 'half', 'max_model_len': 6144, 'tensor_parallel_size': 2, 'gpu_memory_utilization': 0.9, 'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-7B-Instruct'}
(MainProcess pid=840) INFO 11-24 17:55:03 [model.py:761] Resolved architecture: Qwen2ForCausalLM
(MainProcess pid=840) WARNING 11-24 17:55:03 [model.py:2560] Casting torch.bfloat16 to torch.float16.
(MainProcess pid=840) INFO 11-24 17:55:03 [model.py:2214] Using max model len 6144
(MainProcess pid=840) INFO 11-24 17:55:03 [scheduler.py:354] Chunked prefill is enabled with max_num_batche

In [5]:
# Tóm tắt
import random, shutil
import numpy as np, pandas as pd
from pathlib import Path
pd.set_option("display.width", 250)

ev = pd.read_csv(f"{OUT}/llm_eval.csv").drop_duplicates(["name", "method"], keep="last")
print("LLM chấm:", ", ".join(ev["judge"].dropna().unique()))
lab = {"qudar_llm": "llm paper", "qudar_llm_codeprompt": "llm code", "qudar_llm_paper_fixed": "paper cố định",
       "qudar_llm_code_shuffle": "code xáo", "dat": "DAT"}
cells = {}
for _, r in ev.iterrows():
    cells.setdefault(r["name"], {})[lab[r["method"]]] = f'{r["Recall@10"]:.4f} / {r["nDCG@10"]:.4f}'
for name in cells:
    if "_first" not in name:  # Equal trên tập con chưa có, sẽ tính ở bước kiểm định
        eq = pd.read_csv(f"{RUNS}/{name}/eval_results.csv").set_index("method").loc["qudar_simple_equal"]
        cells[name] = {"Equal": f'{eq["Recall@10"]:.4f} / {eq["nDCG@10"]:.4f}', **cells[name]}
cols = ["Equal", "llm paper", "llm code", "paper cố định", "code xáo", "DAT"]
tab = pd.DataFrame.from_dict(cells, orient="index").reindex(columns=cols)
print("Recall@10 / nDCG@10 (Equal lấy từ Step 10c-1, không dùng LLM)")
print(tab.fillna("").to_string())
bad = ev[ev["failed"] > 0]
print("\nQuery chấm lỗi (dùng trọng số đều):", "không có" if bad.empty else "\n" + bad[["name", "method", "queries", "failed"]].to_string(index=False))

# Điểm trung bình theo vị trí hiển thị (cùng cách tính với Step 10-B1 và 10c-2)
rows = []
for f in sorted(Path(OUT).glob("llm_*.jsonl")):
    mode = next((m for m in ("paper_fixed", "code_shuffle", "paper", "code") if f.name.startswith(f"llm_{m}_")), None)
    if mode is None:
        continue
    pos = {i: [] for i in range(1, 5)}
    for line in f.read_text().splitlines():
        r = json.loads(line)
        if not r["scores"]:
            continue
        order = ["ES", "OS", "ED", "OD"]
        if mode in ("paper", "code_shuffle"):
            order = ["OS", "OD", "ES", "ED"]
            random.Random(r["qid"]).shuffle(order)
        for i, s in enumerate(order, 1):
            pos[i].append(r["scores"][s])
    rows.append({"file": f.stem, "n": len(pos[1]), **{f"pos {i}": round(float(np.mean(v)), 2) for i, v in pos.items()}})
print("\n" + pd.DataFrame(rows).to_string(index=False))

dst = Path("/kaggle/working/for_repo/results/step10/llm_qwen")
shutil.copytree(OUT, dst, dirs_exist_ok=True)
shutil.make_archive("/kaggle/working/for_repo_step10c3", "zip", "/kaggle/working/for_repo")
!du -sh {OUT} /kaggle/working/for_repo_step10c3.zip

LLM chấm: Qwen/Qwen2.5-7B-Instruct
Recall@10 / nDCG@10 (Equal lấy từ Step 10c-1, không dùng LLM)
                                Equal        llm paper         llm code    paper cố định         code xáo              DAT
scifact_test          0.8649 / 0.7231  0.8693 / 0.7174  0.8593 / 0.7256  0.8668 / 0.7195  0.8726 / 0.7330  0.8466 / 0.7249
nfcorpus_test         0.1726 / 0.3528  0.1724 / 0.3459  0.1730 / 0.3508  0.1749 / 0.3478  0.1758 / 0.3579  0.1529 / 0.3281
dbpedia-entity_test   0.2595 / 0.3978  0.2500 / 0.4009  0.2502 / 0.4055  0.2518 / 0.3966  0.2554 / 0.4066  0.2008 / 0.3181
fever_test            0.7901 / 0.6357  0.7830 / 0.6419                                                     0.6526 / 0.4898
fever_test_first1500                   0.7819 / 0.6471  0.7755 / 0.6477  0.7678 / 0.6297  0.7857 / 0.6590                 

Query chấm lỗi (dùng trọng số đều): không có

                                file    n  pos 1  pos 2  pos 3  pos 4
        llm_code_dbpedia-entity_test  400   2.86